# OmniFall 1 — Annotations

OmniFall unifies ten fall-detection video datasets under one 16-class taxonomy.
This notebook uses only the annotations, which live on the HuggingFace Hub and
are a few MB. No videos and no downloads beyond that.

```bash
pip install omnifall
```

In [ ]:
import omnifall

omnifall.__version__

## Configs

A *config* is one way of slicing OmniFall: which component datasets it draws
from, and how they are split. The Hub is queried live, so configs added after
this package was released show up too.

In [ ]:
configs = omnifall.list_configs()
print(f"{len(configs)} configs")
configs[:12]

The naming is systematic:

| Family | Example | Meaning |
|---|---|---|
| annotations only | `labels` | every staged + in-the-wild segment, no split |
| same-domain | `le2i-cs` | one component, cross-**s**ubject split |
| | `le2i-cv` | one component, cross-**v**iew split |
| aggregate | `cs`, `cv` | all staged + in-the-wild together |
| synthetic | `of-syn` | the generated component |
| cross-domain | `of-sta-to-all-cs` | train on staged, test on everything |

Some older names are kept as aliases so existing code keeps working:

In [ ]:
list(omnifall.DEPRECATED_CONFIGS.items())[:6]

## Loading

`omnifall.load` wraps `datasets.load_dataset`, so what comes back is an ordinary
`Dataset` / `DatasetDict` and everything you know about 🤗 `datasets` applies.

In [ ]:
ds = omnifall.load("le2i-cs")
ds

Each row is one **temporal segment** of one video — not a whole video. A single
file usually carries many segments.

In [ ]:
train = ds["train"]
row = train[0]
row

| Column | Meaning |
|---|---|
| `path` | video path, relative to its own dataset, **no extension** |
| `dataset` | which component the video belongs to |
| `label` | class id 0–15 |
| `start`, `end` | segment bounds in seconds |
| `subject`, `cam` | subject and camera id, `-1` where not applicable |

The pair `(dataset, path)` is what addresses a video file. Nothing keys off the
config name.

In [ ]:
print("segments:", len(train))
print("distinct videos:", len(set(train["path"])))
print("components:", set(train["dataset"]))

## The label space

`label` is a 🤗 `ClassLabel`, so it converts to and from names.

In [ ]:
feat = train.features["label"]
print(feat.int2str(row["label"]))
print(omnifall.ACTIVITY_LABELS)

OmniFall separates the fall *event* from the *fallen* state, which is the
distinction most single-dataset benchmarks collapse.

In [ ]:
import collections

counts = collections.Counter(train["label"])
for idx, n in counts.most_common():
    print(f"{omnifall.IDX2LABEL[idx]:>10s}  {n:5d}")

The distribution is heavily skewed, which is why accuracy alone is a poor metric
here — see notebook 3.

## Working across components

The aggregate configs mix several components in one table.

In [ ]:
agg = omnifall.load("cs", split="test")
collections.Counter(agg["dataset"]).most_common()

Standard 🤗 `datasets` operations work as usual:

In [ ]:
falls = agg.filter(lambda r: r["label"] == omnifall.LABEL2IDX["fall"])
print(len(falls), "fall segments")
print("mean duration: %.2f s" % (
    sum(r["end"] - r["start"] for r in falls) / len(falls)
))

## Component datasets

In [ ]:
for name, info in omnifall.DATASETS.items():
    print(f"{name:11s} {info.kind:7s} {info.n_videos:6d} videos   {info.homepage}")

## Next

- **[02_videos.ipynb](02_videos.ipynb)** — getting the video files and decoding segments
- **[03_training.ipynb](03_training.ipynb)** — training a model on them